# HKBU Study Companion - Minimum Implementation Baseline

This notebook is a project-owned, end-to-end baseline for COMP4146/7125 minimum implementation checks.

It demonstrates:
- Baseline generation (No RAG)
- Lexical RAG (TF-IDF)
- Optional embedding RAG
- Token usage comparison and a simple quality proxy comparison


In [ ]:
from pathlib import Path
import re
import sys

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").exists():
    pass
elif (PROJECT_ROOT / "7125_Companion_Ass-main" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT / "7125_Companion_Ass-main"
else:
    raise RuntimeError("Cannot locate project root with src/ directory.")

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from hkbu_study_companion.pipeline import StudyCompanion

print("Project root:", PROJECT_ROOT)
print("Src path:", SRC_PATH)


In [ ]:
MODEL = "gemma3:4b"
DOCS_JSON = PROJECT_ROOT / "data" / "docs.example.json"

QUERIES = [
    "What are the core topics in COMP4146?",
    "What is the late submission policy?",
    "Give me a one-week study plan for COMP4146."
]

print("Docs file exists:", DOCS_JSON.exists(), DOCS_JSON)


In [ ]:
def total_tokens(case: dict) -> int:
    stats = case.get("token_stats") or {}
    return int(stats.get("prompt_eval_count") or 0) + int(stats.get("eval_count") or 0)


def quality_proxy(case: dict) -> dict:
    answer = str(case.get("answer") or "")
    return {
        "answer_len": len(answer),
        "citations": len(re.findall(r"\[C\d+\]", answer)),
        "uncertain": any(k in answer.lower() for k in ["uncertain", "not found", "insufficient"]),
    }


In [ ]:
companion = StudyCompanion(
    docs_json=str(DOCS_JSON),
    model=MODEL,
    top_k=4,
    temperature=0.2,
    top_p=0.9,
    num_predict=220,
)

for i, q in enumerate(QUERIES, start=1):
    print("\n" + "#" * 90)
    print(f"Query {i}: {q}")

    baseline = companion.answer_baseline(q)
    tfidf = companion.answer_tfidf(q)

    print("\n[Baseline]", baseline.get("token_stats"), quality_proxy(baseline))
    print(baseline.get("answer", "")[:500])

    print("\n[TF-IDF]", tfidf.get("token_stats"), quality_proxy(tfidf))
    print(tfidf.get("answer", "")[:500])

    print("\nToken delta (TF-IDF - Baseline):", total_tokens(tfidf) - total_tokens(baseline))

    try:
        embed = companion.answer_embed(q, embed_model="sentence-transformers/all-MiniLM-L6-v2")
        print("\n[Embedding]", embed.get("token_stats"), quality_proxy(embed))
        print(embed.get("answer", "")[:500])
        print("Token delta (Embedding - Baseline):", total_tokens(embed) - total_tokens(baseline))
    except Exception as e:
        print("\n[Embedding] skipped:", e)


## How To Use In Report

You can cite this notebook as evidence for minimum implementations:
- End-to-end baseline in Jupyter Notebook
- Baseline vs RAG comparisons
- Token usage comparison
- Basic quality comparison (proxy)
